# CIC-IDS2017 — DQN Reinforcement Learning Pipeline
## Phase 1 (Part 2): Train RL Agent on the same CIC-IDS2017 data

**Purpose:** Train a Deep Q-Network (DQN) firewall agent on the same  
stratified 10 % sample of CIC-IDS2017 used in the ML notebook,  
so results are directly comparable with Random Forest & Gradient Boosting.

**MDP formulation (from your papers):**
| Component | Definition |
|---|---|
| **State** | Feature vector of one network flow (scaled) |
| **Actions** | 0 = Allow (Benign), 1 = Block (Attack) |
| **Reward** | +1 correct, −1 wrong, −0.5 false positive |
| **Algorithm** | DQN with Experience Replay + Target Network |


## 1 · System Check

In [4]:
import platform, psutil, os

print("=" * 55)
print("  SYSTEM INFORMATION")
print("=" * 55)
print(f"  OS     : {platform.system()} {platform.release()}")
print(f"  CPU    : {os.cpu_count()} logical cores")
ram = psutil.virtual_memory()
print(f"  RAM    : {ram.total/1e9:.1f} GB total  |  {ram.available/1e9:.1f} GB free")

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  PyTorch: {torch.__version__}")
print(f"  Device : {device}  {'(GPU — fast!)' if device.type=='cuda' else '(CPU — fine for this)'}")
print("=" * 55)


  SYSTEM INFORMATION
  OS     : Windows 11
  CPU    : 8 logical cores
  RAM    : 16.9 GB total  |  4.2 GB free
  PyTorch: 2.9.1+cpu
  Device : cpu  (CPU — fine for this)


## 2 · Imports

In [6]:
import pandas as pd
import numpy as np
import warnings, pickle, time, json, gc, random
from pathlib import Path
from collections import deque
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score, matthews_corrcoef
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ All imports OK  |  device = {device}")


✓ All imports OK  |  device = cpu


## 3 · Data Loading — Same Stratified 10 % Sample

> We load **exactly the same data** as the ML notebook  
> (same `DATA_DIR`, same `SAMPLE_FRAC`, same `RANDOM_STATE = 42`).  
> This is what makes the comparison between RL and RF/GB fair.


In [8]:
# ── ① Change this to your folder (same as ML notebook) ───────────────────────
DATA_DIR    = Path(r'C:/Users/DELL/Desktop/Graduation Project/GP_Models/data/TrafficLabelling')
SAMPLE_FRAC = 0.10
# ─────────────────────────────────────────────────────────────────────────────

csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f"Found {len(csv_files)} CSV files")

chunks = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f, encoding='cp1252', low_memory=False)
        tmp.columns = tmp.columns.str.strip()
        label_col = next((c for c in ['Label','label','Class','class']
                          if c in tmp.columns), tmp.columns[-1])
        sampled = (tmp.groupby(label_col, group_keys=False)
                      .apply(lambda x: x.sample(frac=SAMPLE_FRAC,
                                                 random_state=RANDOM_STATE)))
        chunks.append(sampled)
        del tmp; gc.collect()
        print(f"  ✓ {f.name}: {len(sampled):,} rows")
    except Exception as e:
        print(f"  ✗ {f.name}: {e}")

df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()
print(f"\n✓ Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")


Found 8 CSV files
  ✓ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 22,575 rows
  ✓ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 28,647 rows
  ✓ Friday-WorkingHours-Morning.pcap_ISCX.csv: 19,104 rows
  ✓ Monday-WorkingHours.pcap_ISCX.csv: 52,992 rows
  ✓ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 28,861 rows
  ✓ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 17,037 rows
  ✓ Tuesday-WorkingHours.pcap_ISCX.csv: 44,591 rows
  ✓ Wednesday-workingHours.pcap_ISCX.csv: 69,270 rows

✓ Dataset: 283,077 rows × 85 columns


## 4 · Cleaning & Target Variable

In [11]:
from sklearn.preprocessing import LabelEncoder

# Identify label column
LABEL_COL = next((c for c in ['Label','label','Class','class']
                  if c in df.columns), df.columns[-1])

# Clean column names
df.columns = (df.columns.str.strip()
                .str.replace(' ', '_', regex=False)
                .str.lower())
LABEL_COL = LABEL_COL.strip().lower().replace(' ','_')

# Binary target
df['target'] = (~df[LABEL_COL].str.strip().str.upper()
                               .isin(['BENIGN','NORMAL','LEGITIMATE'])).astype(np.int8)

print(f"Benign  (0): {(df['target']==0).sum():,}  ({(df['target']==0).mean()*100:.1f} %)")
print(f"Attack  (1): {(df['target']==1).sum():,}  ({(df['target']==1).mean()*100:.1f} %)")

# Handle infinities / NaN
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# Encode categorical columns
obj_cols = [c for c in df.select_dtypes(include='object').columns
            if c not in [LABEL_COL, 'target']]
for c in obj_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str)).astype(np.int32)

# Drop irrelevant columns
drop_kw = ['timestamp','flowstarttime','flowendtime',
           'src_ip','dst_ip','sourceip','destip', LABEL_COL]
to_drop  = [c for c in df.columns
            if any(k in c for k in drop_kw) and c != 'target']
df.drop(columns=list(set(to_drop)), inplace=True, errors='ignore')

# Memory optimisation
f64 = df.select_dtypes('float64').columns
i64 = df.select_dtypes('int64').columns
df[f64] = df[f64].astype(np.float32)
df[i64] = df[i64].astype(np.int32)

mem = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nFinal shape: {df.shape}   RAM: {mem:.0f} MB")


Benign  (0): 227,311  (80.3 %)
Attack  (1): 55,766  (19.7 %)

Final shape: (283077, 84)   RAM: 90 MB


## 5 · Train / Test Split & Normalisation

In [13]:
X = df.drop(columns=['target']).values.astype(np.float32)
y = df['target'].values.astype(np.int64)
FEATURE_NAMES = df.drop(columns=['target']).columns.tolist()
FEATURE_DIM = X.shape[1]

print(f"Features: {FEATURE_DIM}")
print(f"Samples : {len(y):,}")

# ── Fix: drop exact duplicate rows BEFORE splitting ──────────────────────────
df_temp = pd.DataFrame(X)
df_temp['target'] = y
df_temp = df_temp.drop_duplicates()
X = df_temp.drop(columns=['target']).values.astype(np.float32)
y = df_temp['target'].values.astype(np.int64)
print(f"After dedup: {len(y):,} rows  (removed {len(df['target']) - len(y):,} duplicates)")
del df_temp

# ── Stratified split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

# ── Verify no leakage ────────────────────────────────────────────────────────
train_set = set(map(tuple, X_train[:100].astype(np.float16)))
test_set  = set(map(tuple, X_test[:100].astype(np.float16)))
overlap   = train_set & test_set
print(f"Overlap check (should be 0): {len(overlap)}")

# ── Normalise ────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc  = scaler.transform(X_test).astype(np.float32)

X_train_sc = pd.DataFrame(X_train_sc, columns=FEATURE_NAMES)
X_test_sc  = pd.DataFrame(X_test_sc,  columns=FEATURE_NAMES)

del X
gc.collect()
print(f"\nTrain : {len(X_train_sc):,}  |  Test : {len(X_test_sc):,}")
print("✓ Split and scaling done")

Features: 83
Samples : 283,077
After dedup: 282,781 rows  (removed 296 duplicates)
Overlap check (should be 0): 0

Train : 197,946  |  Test : 84,835
✓ Split and scaling done


## 6 · DQN Architecture

The network takes a **state vector** (one normalised network flow) and outputs  
**Q-values** for two actions: Allow (0) and Block (1).

Architecture follows the FireRL / MDPI DQN papers:  
Input → 256 → 128 → 64 → 2 outputs


In [16]:
class DQN(nn.Module):
    """
    Deep Q-Network for firewall policy.
    Input  : state vector  (1 × FEATURE_DIM)
    Output : Q-values      (1 × 2)  [Q_allow, Q_block]
    """
    def __init__(self, state_dim: int, action_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)


# Quick sanity check
dummy = torch.zeros(1, FEATURE_DIM)
model_test = DQN(FEATURE_DIM)
out = model_test(dummy)
print(f"✓ DQN architecture OK")
print(f"  Input  : {FEATURE_DIM} features")
print(f"  Output : {out.shape[1]} Q-values  [Q_allow, Q_block]")

total_params = sum(p.numel() for p in model_test.parameters())
print(f"  Params : {total_params:,}")
del model_test


✓ DQN architecture OK
  Input  : 83 features
  Output : 2 Q-values  [Q_allow, Q_block]
  Params : 62,786


## 7 · Experience Replay Buffer

In [18]:
class ReplayBuffer:
    """
    Stores (state, action, reward, next_state, done) transitions.
    Randomly samples mini-batches to break temporal correlations —
    a key technique in DQN (DeepMind, 2015).
    """
    def __init__(self, capacity: int = 50_000):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(ns)).to(device),
                torch.FloatTensor(d).to(device))

    def __len__(self): return len(self.buf)


print("✓ ReplayBuffer ready  (capacity = 50,000 transitions)")


✓ ReplayBuffer ready  (capacity = 50,000 transitions)


## 8 · DQN Agent

Key design choices (all from your papers):
- **ε-greedy exploration** decaying from 1.0 → 0.05
- **Target network** updated every 200 steps (stabilises training)
- **Prioritised reward** +1 correct block / correct allow, −1 missed attack (false negative), −0.5 false alarm (false positive)


In [20]:
class DQNAgent:
    def __init__(self, state_dim, action_dim=2,
                 lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_capacity=50_000, batch_size=256,
                 target_update_freq=200):

        self.action_dim        = action_dim
        self.gamma             = gamma
        self.epsilon           = epsilon_start
        self.epsilon_end       = epsilon_end
        self.epsilon_decay     = epsilon_decay
        self.batch_size        = batch_size
        self.target_update_freq = target_update_freq
        self.steps             = 0

        # Online network (trained every step)
        self.policy_net = DQN(state_dim, action_dim).to(device)
        # Target network (updated periodically — stabilises Q-targets)
        self.target_net = DQN(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory    = ReplayBuffer(buffer_capacity)

    # ── action selection (ε-greedy) ──────────────────────────────────────────
    def select_action(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(device)
            return int(self.policy_net(s).argmax(dim=1).item())

    # ── reward shaping (from FireRL / MDPI papers) ────────────────────────────
    @staticmethod
    def compute_reward(action: int, true_label: int) -> float:
        if action == true_label:
            return +1.0                  # correct decision
        elif action == 0 and true_label == 1:
            return -1.0                  # missed attack (false negative) — worst
        else:
            return -0.5                  # false alarm  (false positive) — annoying

    # ── store transition ──────────────────────────────────────────────────────
    def store(self, s, a, r, ns, done):
        self.memory.push(s, a, r, ns, done)

    # ── training step ─────────────────────────────────────────────────────────
    def train_step(self) -> float:
        if len(self.memory) < self.batch_size:
            return 0.0

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        # Current Q-values for taken actions
        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q-values (Bellman)
        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1)[0]
            targets    = rewards + self.gamma * max_next_q * (1 - dones)

        loss = F.mse_loss(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

        # Periodically sync target network
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return loss.item()


agent = DQNAgent(state_dim=FEATURE_DIM)
print("✓ DQN Agent ready")
print(f"  Policy net params : {sum(p.numel() for p in agent.policy_net.parameters()):,}")
print(f"  Learning rate     : 1e-3")
print(f"  Gamma (discount)  : 0.99")
print(f"  Epsilon start→end : 1.0 → 0.05  (decay 0.995)")
print(f"  Batch size        : 256")
print(f"  Target update     : every 200 steps")


✓ DQN Agent ready
  Policy net params : 62,786
  Learning rate     : 1e-3
  Gamma (discount)  : 0.99
  Epsilon start→end : 1.0 → 0.05  (decay 0.995)
  Batch size        : 256
  Target update     : every 200 steps


## 9 · Training Loop

> **How this works:**  
> The agent sees one flow at a time (state), decides Allow/Block (action),  
> gets a reward signal, and stores the experience.  
> Every step it samples a random mini-batch and updates its neural network.  
> Over many episodes it learns which features indicate an attack.

> **⏱ Expected time:** ~5–15 min on CPU depending on your laptop.


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
N_EPISODES   = 10       # one episode = one pass through the training set
LOG_INTERVAL = 2        # print metrics every N episodes
# ─────────────────────────────────────────────────────────────────────────────

train_size  = len(X_train)
history     = {'episode': [], 'loss': [], 'reward': [],
               'accuracy': [], 'f1': [], 'epsilon': []}

print(f"Training DQN for {N_EPISODES} episodes  |  {train_size:,} steps per episode")
print(f"Device: {device}")
print("=" * 65)

t_start = time.time()

for episode in range(1, N_EPISODES + 1):
    ep_loss, ep_reward = 0.0, 0.0
    preds, trues       = [], []

    # Shuffle training data each episode
    idx = np.random.permutation(train_size)

    for i in idx:
        state      = X_train[i]
        true_label = int(y_train[i])

        # Agent decides
        action = agent.select_action(state)
        reward = agent.compute_reward(action, true_label)

        # Next state is just the next flow (dataset is not sequential —
        # we use done=True so no bootstrapping across flows)
        next_state = X_train[(i + 1) % train_size]
        agent.store(state, action, float(reward), next_state, 1.0)

        loss = agent.train_step()

        ep_loss   += loss
        ep_reward += reward
        preds.append(action)
        trues.append(true_label)

    acc = np.mean(np.array(preds) == np.array(trues))
    f1  = f1_score(trues, preds, zero_division=0)

    history['episode'].append(episode)
    history['loss'].append(ep_loss / train_size)
    history['reward'].append(ep_reward / train_size)
    history['accuracy'].append(acc)
    history['f1'].append(f1)
    history['epsilon'].append(agent.epsilon)

    elapsed = time.time() - t_start
    remaining = elapsed / episode * (N_EPISODES - episode)

    if episode % LOG_INTERVAL == 0 or episode == 1:
        print(f"  Ep {episode:2d}/{N_EPISODES}  "
              f"Acc={acc:.4f}  F1={f1:.4f}  "
              f"Reward={ep_reward/train_size:+.3f}  "
              f"ε={agent.epsilon:.3f}  "
              f"[{elapsed/60:.1f}m elapsed, ~{remaining/60:.1f}m left]")

print("=" * 65)
print(f"\n✓ Training complete in {(time.time()-t_start)/60:.1f} minutes")


Training DQN for 10 episodes  |  197,946 steps per episode
Device: cpu
  Ep  1/10  Acc=0.7877  F1=0.0447  Reward=+0.586  ε=0.050  [36.6m elapsed, ~329.8m left]


## 10 · Evaluation on Test Set

In [ ]:
print("Evaluating DQN on held-out test set …")
agent.policy_net.eval()

preds_rl  = []
probas_rl = []

with torch.no_grad():
    # Process in batches for speed
    batch_size = 1024
    for start in range(0, len(X_test), batch_size):
        batch = torch.FloatTensor(X_test[start:start+batch_size]).to(device)
        q_vals = agent.policy_net(batch)
        proba  = torch.softmax(q_vals, dim=1)[:, 1]   # P(attack)
        pred   = q_vals.argmax(dim=1)
        preds_rl.extend(pred.cpu().numpy().tolist())
        probas_rl.extend(proba.cpu().numpy().tolist())

preds_rl  = np.array(preds_rl)
probas_rl = np.array(probas_rl)

acc_rl  = (preds_rl == y_test).mean()
prec_rl = precision_score(y_test, preds_rl, zero_division=0)
rec_rl  = recall_score(y_test, preds_rl, zero_division=0)
f1_rl   = f1_score(y_test, preds_rl, zero_division=0)
auc_rl  = roc_auc_score(y_test, probas_rl)
mcc_rl  = matthews_corrcoef(y_test, preds_rl)

print("=" * 55)
print("  DQN RL AGENT — TEST SET PERFORMANCE")
print("=" * 55)
print(classification_report(y_test, preds_rl,
                             target_names=['Benign','Attack']))
print(f"  Accuracy    : {acc_rl:.4f}")
print(f"  Precision   : {prec_rl:.4f}")
print(f"  Recall      : {rec_rl:.4f}")
print(f"  F1-Score    : {f1_rl:.4f}")
print(f"  ROC-AUC     : {auc_rl:.4f}")
print(f"  Matthews CC : {mcc_rl:.4f}")


## 11 · Training Curves & Evaluation Plots

In [ ]:
cm_rl = confusion_matrix(y_test, preds_rl)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('DQN RL Agent — CIC-IDS2017', fontsize=14, fontweight='bold')

# 1 · Accuracy over episodes
ax = axes[0, 0]
ax.plot(history['episode'], history['accuracy'], marker='o', color='#3498db', lw=2)
ax.set_title('Accuracy over Episodes'); ax.set_xlabel('Episode')
ax.set_ylabel('Accuracy'); ax.set_ylim([0, 1.05]); ax.grid(alpha=0.3)

# 2 · F1 over episodes
ax = axes[0, 1]
ax.plot(history['episode'], history['f1'], marker='s', color='#2ecc71', lw=2)
ax.set_title('F1-Score over Episodes'); ax.set_xlabel('Episode')
ax.set_ylabel('F1'); ax.set_ylim([0, 1.05]); ax.grid(alpha=0.3)

# 3 · Epsilon decay
ax = axes[0, 2]
ax.plot(history['episode'], history['epsilon'], marker='^', color='#e67e22', lw=2)
ax.set_title('ε Decay (Exploration → Exploitation)')
ax.set_xlabel('Episode'); ax.set_ylabel('Epsilon'); ax.grid(alpha=0.3)

# 4 · Confusion matrix
ax = axes[1, 0]
sns.heatmap(cm_rl, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Benign','Attack'], yticklabels=['Benign','Attack'])
ax.set_title('Confusion Matrix'); ax.set_ylabel('True'); ax.set_xlabel('Predicted')

# 5 · ROC curve
ax = axes[1, 1]
fpr, tpr, _ = roc_curve(y_test, probas_rl)
ax.plot(fpr, tpr, lw=2, color='purple', label=f'DQN AUC={auc_rl:.4f}')
ax.plot([0,1],[0,1],'--', color='grey', label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curve')
ax.legend(); ax.grid(alpha=0.3)

# 6 · Average reward
ax = axes[1, 2]
ax.plot(history['episode'], history['reward'], marker='D', color='#e74c3c', lw=2)
ax.axhline(0, color='grey', ls='--', lw=1)
ax.set_title('Avg Reward per Step'); ax.set_xlabel('Episode')
ax.set_ylabel('Reward'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('rl_training_results.png', dpi=100)
plt.show()
print("✓ Saved: rl_training_results.png")


## 12 · Phase 1 Comparison: RF vs GB vs DQN

In [ ]:
# Load ML results from metadata file saved by the ML notebook
META_PATH = Path('outputs/metadata.json')

if META_PATH.exists():
    with open(META_PATH) as fh:
        meta = json.load(fh)
    rf_f1  = meta['RF']['f1'];        rf_auc  = meta['RF']['roc_auc']
    gb_f1  = meta['GB_tuned']['f1'];  gb_auc  = meta['GB_tuned']['roc_auc']
    rf_rec = meta['RF'].get('recall', 'N/A')
    gb_rec = meta['GB_tuned'].get('recall', 'N/A')
else:
    print("⚠ outputs/metadata.json not found — run the ML notebook first.")
    print("  Using placeholder values for the comparison chart.")
    rf_f1 = rf_auc = gb_f1 = gb_auc = rf_rec = gb_rec = 0.0

print("=" * 65)
print("  PHASE 1 COMPARISON — CIC-IDS2017 (same 10 % stratified sample)")
print("=" * 65)
print(f"  {'Model':<22}  {'F1-Score':>9}  {'ROC-AUC':>9}  {'Recall':>9}")
print("-" * 65)
print(f"  {'Random Forest':<22}  {rf_f1:>9.4f}  {rf_auc:>9.4f}  {str(rf_rec):>9}")
print(f"  {'Gradient Boosting':<22}  {gb_f1:>9.4f}  {gb_auc:>9.4f}  {str(gb_rec):>9}")
print(f"  {'DQN RL Agent':<22}  {f1_rl:>9.4f}  {auc_rl:>9.4f}  {rec_rl:>9.4f}")
print("=" * 65)
print("\n→ NEXT: Run Phase 2 notebook to test all three on NSL-KDD")
print("  (concept-drift test — new attack types not seen during training)")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
models  = ['Random Forest', 'Gradient Boosting', 'DQN (RL)']
f1s     = [rf_f1, gb_f1, f1_rl]
aucs    = [rf_auc, gb_auc, auc_rl]
x = np.arange(3); w = 0.35
bars1 = ax.bar(x - w/2, f1s,  w, label='F1-Score', color='#3498db')
bars2 = ax.bar(x + w/2, aucs, w, label='ROC-AUC',  color='#9b59b6')
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylim([0.6, 1.02])
ax.set_title('Phase 1: Model Comparison on CIC-IDS2017', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3, axis='y')
for b in bars1: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                         f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for b in bars2: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                         f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('phase1_comparison.png', dpi=100)
plt.show()
print("✓ Saved: phase1_comparison.png")


## 13 · Save Model Artefacts

In [ ]:
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

# Save DQN weights (PyTorch)
torch.save(agent.policy_net.state_dict(), OUT / 'dqn_policy_net.pth')

# Save full agent config
with open(OUT / 'scaler_rl.pkl', 'wb') as fh:
    pickle.dump(scaler, fh)

rl_meta = {
    'model_type'   : 'DQN (Deep Q-Network)',
    'feature_dim'  : int(FEATURE_DIM),
    'train_rows'   : int(len(X_train)),
    'test_rows'    : int(len(X_test)),
    'n_episodes'   : N_EPISODES,
    'final_epsilon': round(float(agent.epsilon), 4),
    'performance'  : {
        'accuracy' : round(float(acc_rl),  4),
        'precision': round(float(prec_rl), 4),
        'recall'   : round(float(rec_rl),  4),
        'f1'       : round(float(f1_rl),   4),
        'roc_auc'  : round(float(auc_rl),  4),
        'mcc'      : round(float(mcc_rl),  4)
    },
    'training_history': history
}

with open(OUT / 'rl_metadata.json', 'w') as fh:
    json.dump(rl_meta, fh, indent=2)

print("✓ Saved to outputs/")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size/1024:.1f} KB)")


## 14 · Summary & MDP Explanation for your Report

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════╗
║           DQN RL AGENT — PHASE 1 COMPLETE                    ║
╠═══════════════════════════════════════════════════════════════╣""")
print(f"""║  Accuracy  : {acc_rl:.4f}                                        ║
║  Precision : {prec_rl:.4f}                                        ║
║  Recall    : {rec_rl:.4f}                                        ║
║  F1-Score  : {f1_rl:.4f}                                        ║
║  ROC-AUC   : {auc_rl:.4f}                                        ║
╠═══════════════════════════════════════════════════════════════╣
║  MDP FORMULATION (for your report):                          ║
║  • State  : scaled network flow feature vector               ║
║  • Action : 0 = Allow traffic   1 = Block traffic            ║
║  • Reward : +1 correct  |  -1 missed attack  |  -0.5 FP      ║
║  • Algo   : DQN + Experience Replay + Target Network         ║
║  • Epsilon: greedy decay 1.0 → 0.05 over episodes            ║
╠═══════════════════════════════════════════════════════════════╣
║  NEXT STEP → Phase 2: test RF, GB, DQN on NSL-KDD            ║
║  (concept-drift: new attack types not seen in training)      ║
╚═══════════════════════════════════════════════════════════════╝""")
